Un **sistema software auto-adattivo** è capace di modificare il proprio funzionamento **runtime** in risposta a cambiamenti:
- nel proprio **stato interno**
- **nell'ambiente esterno** in cui opera

L'obiettivo di questi sistemi è continuare a fornire il servizio e rispettare determinati requisiti, richiedendo un intervento umano limitato (es. un'applicazione cloud può aumentare il numero di istanze in risposta a un aumento del carico di lavoro).

Si collegano questi sistemi al concetto più generale di **autonomic computing**, che a sua volta è un paradigma ispirato al sistema nervoso umano per cui alcune funzioni corporee (battito cardiaco, respirazione, digestione) sono regolate automaticamente senza intervento cosciente.  
Analogamente un sistema informatico autonomo gestisce da solo alcuni aspetti del proprio funzionamento. 

Chiaramente gli **obiettivi e le regole devono essere definite dai progettisti o dagli amministratori del sistema**.  
I principi di auto-adattamento si applicano in molti contesti, a noi interessano in particolare applicati al cloud, all'edge computing e al computing continuum.

Alcuni esempi di sistemi software auto-adattivi:

<img src="imgs/auto.png" width="400">

es. Kubertenes è un sistema di orchestrazione che gestisce applicazioni eseguite in container su un insieme di nodi. Tra le attività di adattamento si hanno ad es. placement o scheduling (decidere su quale nodo eseguire un container), scaling orizzontale (aumentare o diminuire il numero di istanze di un'applicazione), verticale (aumentare o diminuire le risorse per un singolo componente) e healing (ricreare su nodi disponibili componenti che erano eseguiti in un nodo ora guasto).

es. Amazon EC2 Auto Scaling è un servizio che permette di aumentare o diminuire automaticamente il numero di istanze EC2 in base a regole definite dall'utente, come ad esempio l'utilizzo della CPU o il numero di richieste in coda. **Il sistema adatta automaticamente le risorse per mantenere il comportamento desiderato**.

Si citano anche tra gli esempi reti che si riconfigurano per reagire ai guasti (l'abbiamo visto in dettaglio con Chord) e database capaci di gestire automaticamente le risorse.

## self-* capabilities

Esistono diverse capacità di autogestione, raggruppate nella notazione **self-{*}**:
1. **Self-Optimization**: ottimizzare prestazioni o uso delle risorse, rispettando gli obiettivi di qualità (es. cambiare il posizionamento dei componenti per ridurre i tempi di risposta);
2. **Self-Healing**: rilevare, diagnosticare e recuperare da guasti (es. escludere un nodo guasto dal bilanciamento del carico);
3. **Self-Configuration**: integrare nuovi elementi o modificare automaticamente parametri di configurazione (es. aggiungere un nuovo nodo al servizio);
4. **Self-Protection**: rilevare minacce e reagire per limitare gli effetti (es. identificare un attacco DDoS e bloccare il traffico malevolo).

Per raggiungere questi obiettivi, il sistema deve:
- **conoscere il proprio stato interno (self-awareness)** quindi risorse disponibili, prestazioni, componenti attivi etc..
- **conoscere il contesto operativo (self-situation)** quindi condizioni della rete, utenti, carico esterno e ambiente
- **rilevare i cambiamenti (self-monitoring)**, osservare cioè come evolvono sistema e contesto
- **intervenire (self-adjustement)**, modificando il proprio comportamento in risposta a ciò che è stato rilevato.

Concettualmente il modello di un sistema self-adaptive è fatto di due parti:
- **Managed system**: è il software che fornisce il servizio e che deve essere adattato (es. un'applicazione Web con più istanze)
- **Managing system**: è il sistema di gestione vero e proprio, che osserva il managed system e decide come adattarlo per perseguire gli obiettivi prestabiliti.

Tra i due sistemi, serve una strumentazione che permetta di **misurare** (attraverso sensori o sonde) e **intervenire** nel caso in cui sia necessario adattare il sistema attraverso meccanismi di intervento specifici (**effettori**).

Il managing system, oltre che il managed system, deve osservare anche l'ambiente in il managed system si trova (quindi ciò che non è sotto il suo controllo diretto come utenti, rete, software esterno etc..)

<img src="imgs/self-adaptive.png" width="600">

es. in un'applicazione cloud il sistema gestito serve le richieste; le sonde misurano il carico, la CPU e i tempi di risposta; il sistema di gestione valuta se le risorse sono sufficienti gli effettori possono creare nuove repliche

Si tratta di un così detto **ciclo di controllo a retroazione (feedback loop)**. 

## MAPE-K
Questo ciclo è organizzato nel modello MAPE:
$$ \text{Monitor} \rightarrow \text{Analyze} \rightarrow \text{Plan} \rightarrow \text{Execute}$$
Queste quattro fasi usano una base di conoscenza, **Knowledge**, da cui la denominazione MAPE-K.

In generale:
- **Monitor** raccoglie i dati dal sistema e dall'ambiente. Può filtrarli, aggregarli e correrarli per ottenere informazioni più utili da analizzare;
- **Analyze** valuta le informazioni raccolte e determina se è necessario un adattamento. Può usare modelli predittivi, regole o tecniche di machine learning;
- **Plan** è il cuore decisionale del ciclo: si sceglie quale azione effettuare per ottenere il cambiamento desiderato.
- **Execute** applica le azioni pianificate al sistema gestito, usando gli **effettori**.
- **Knowledge**: conserva la conoscenza condivisa necessaria per le varie fasi (es. dati di monitoraggio, stato e caratteristiche del sistema, informazioni sull'ambiente e obiettivi dell'amministratore. Non deve essere per forza un unico database centrale, è una componente logica).

es. tempi di risposta troppo alti --> il monitor raccoglie i tempi di risposta e l'utilizzo delle risorse, l'analyze si accorge che il servizio potrebbe non rispettare gli obiettivi di qualità, il plan decide quante repliche aggiungere e l'execute le avvia.

<img src="imgs/MAPE.png" width="400">

### MAPE: Monitor
Per progettare il monitoraggio ci si devono porre alcune domande:
- **Quando (When) Monitorare?** Il monitoraggio può infatti essere **continuativo** permettendo l'osservazione nel tempo tramite campionamenti regolari, oppure **on-demand** (attivato se richiesto, magari con un evento specifico);
- **Cosa (What) Monitorare?** Si possono monitorare diverse misure:
    - risorse (CPU, memoria..)
    - carico di lavoro (numero di richieste, dimensione dei dati..)
    - prestazioni (tempi di risposta, throughput..)
    - disponibilità e stato dei componenti
- **Come (How) organizzare il monitoraggio?** In un'architettura **centralizzata**, i dati confluiscono in un'entità centrale.  
In una **decentralizzata** la raccolta e l'elaborazione sono distribuite tra più componenti.  
La scelta del **come monitorare** invece è distinta dalla metodologia di misurazione, che può essere:
    - **Passiva**: si osservano i dati o attività già disponibili senza iniettare nuovo traffico nel sistema 
    - **Attiva**: si inviano richieste di misurazione (es. ping, heartbeat, test di carico) per ottenere informazioni più dettagliate (carico aggiuntivo maggiore rispetto al passivo!)
- **Dove (Where) conservare i dati e come prepararli?** es. database a serie temporali come InfluxDB, adatti a misure associate nel tempo come utilizzo della CPU che viene rilevato ogni pochi secondi...  
Sul come prepararli, come anticipato, i dati possono essere filtrati, aggregati, correlati o normalizzati

### MAPE: Analyze
L'analisi come detto usa i dati monitorati per stabilire se serve un adattamento.

La prima scelta riguarda quando attivare l'analisi:
- **a tempo** (periodicamente, es. ogni 2 minuti)
- **a evento** (quando un evento rilevante si verifica, es. un certo numero di richieste in coda)

La seconda distinzione è tra approccio reattivo e proattivo:
- **reattivo**: si reagisce a un cambiamento già avvenuto (es. il carico è aumentato e il sistema rileva che servono più risorse)
- **proattivo**: si prevede la necessità futura di adattamento e si prepara l'intervento in anticipo. Il motivo è che **creare risorse richiede tempo**, e un intervento proattivo può compensare questo ritardo. La previsione può usare tecniche statistiche o di ML (da modelli semplici tipo regressione lineare a reti neurali complesse). Essendo una previsione ovviamente è soggetta a possibili errori.

### MAPE: Plan
Come detto Plan è il cuore decisionale del ciclo, e le sue decisioni devono perseguire **obiettivi ad alto livello** (high-level objectives, si vedono in alto a dx nell'img del modello concettuale) del tipo mantenere il tempo di risposta entro una certa soglia, garantire una certa disponibilità, limitirare il costo delle risorse cloud etc...

Plan è la fase più complessa e studiata, anche perché *le decisioni prese in questa fase devono arrivare in tempo utile: il sistema deve continuare a funzionare mentre il controllore pianifica.*  
Una soluzione ottima ma ottenuta troppo tardi può non essere più utile (si ricorda infatti che MAPE è un ciclo continuo di controllo a retroazione) --> il tempo è vincolo fondamentale.

Per Plan esistono diverse famiglie di metodi:
- **Ottimizzazione matematica**: con questo approccio si definisce il modello **rigorosamente** tramite *variabili decisionali*; una *funzione obiettivo* e *vincoli che le decisioni devono rispettare*.  
Il vantaggio è che la soluzione al modello così costruito è **ottima**, gli svantaggi sono che *se il modello non è accurato rispetto al sistema reale --> l'ottimo matematico può non corrispondere alla decisione migliore in pratica* e che *in molti casi trovare esattamente la soluzione ottima richiede troppo tempo* (infatti molti problemi della gestione delle risorse sono NP-hard, ne vedremo un esempio tra poco). 
- **Euristiche**: un'euristica cerca una soluzione utile con costo computazionale contenuto (è **più veloce**), *non garantendo però che sia ottima*.
- **ML e Reinforcement Learning**: con ML si intende generalmente l'uso di modelli addestrati sui dati.  
Il Reinforcement Learning rappresenta invece un approccio per cui il controllore impara a prendere decisioni attraverso un'interazione continua con il sistema e l'ambiente, ricevendo un feedback (reward) che indica quanto la decisione presa sia stata buona rispetto agli obiettivi ed imparando di conseguenza.  
Anche gli LLM ad oggi sono un possibile strumento di supporto per la fase di Planning (tuttavia attenzione al fatto che una tecnica sofisticata non è necessariamente preferibile: ricorda che il tempo in questo caso è il vincolo fondamentale).
- *Control Theory*: teoria dei sistemi di controllo, si cita ad es. PID (controller proporzionale-integrale-derivativo) che permette di regolare il comportamento del sistema tramite un feedback continuo, stabilizzandosi osservando la differenza tra il comportamento attuale e quello desiderato.
- *Queuing Theory*: modellare il sistema come una rete di code, per cui le richieste attendono prima di essere servite dalle risorse. La teoria delle code permette di analizzare le prestazioni del sistema e prevedere l'impatto di diverse decisioni di adattamento (es. aggiungere risorse, cambiare il posizionamento dei componenti etc..)

#### Esempio concreto: il problema di placement
Il problema di **placement** consiste nel decidere **su quali nodi collocare VM o container che compongono un'applicazione distribuita**, in modo da rispettare gli obiettivi di qualità e ottimizzare l'uso delle risorse (più precisamente, su quali nodi fisici collocare le VM o su quali nodi fisici/VM collocare i container; e ciascun container tipicamente supporta una sola componente dell'applicazione mentre una VM può ospitare $N$ componenti con $N$ numero comunque limitato). 

Questo problema è riconducibile al **bin packing** (correlato al knapsack problem) dove i **bin** sono i **server fisici**, gli **item** le **VM/container**, le *dimensioni dell'item* le *risorse richieste* e la *capacità del bin* le *risorse disponibili* sul server. 

Per risolvere il problema, vogliamo **collocare tutti gli oggetti usando il minor numero di server possibile, senza che si superi la capacità di nessuno di essi**. Infatti chiaramente ridurre il numero di server utilizzati permette di risparmiare risorse (chiaramente qui modello semplificato, nella realtà ci sarebbero da dover considerare disponibilità, comunicazioni e molti altri vincoli..)

<img src="imgs/binpacking.png" width="400">

Formuliamo anzitutto il problema come un modello di ottimizzazione matematica. Siano:
- $I$ insieme di oggetti da collocare (VM/container)
- $n$ numero di server fisici disponibili
- $s_i$ risorse richieste dall'oggetto $i \in I$ (es. CPU, memoria, storage, qui assimilate a un'unica risorsa per semplificare)
- $B$ capacità di ogni server (qui assunta uguale per tutti per semplificare)

Si introducono *le variabili decisionali*:
$$y_j = \begin{cases} 1 & \text{se il server } j \text{ viene utilizzato} \\ 0 & \text{altrimenti} \end{cases} \quad \text{e} \quad x_{ij} = \begin{cases} 1 & \text{se l'oggetto } i \text{ è collocato sul server } j \\ 0 & \text{altrimenti} \end{cases}$$
La *funzione obiettivo*, in quanto si vuol minimizzare il numero di server usati, è:
$$K = \min \sum_{j=1}^{n} y_j$$
I *vincoli* su $K \ge 1$ sono:
1. **ogni oggetto deve essere collocato su un solo server**:
   $$\sum_{j=1}^{n} x_{ij} = 1 \quad \forall i \in I$$
2. **rispettare le capacità dei server, e usare solo quelli attivi**:  
   $$\sum_{i \in I} s_i x_{ij} \leq B y_j \quad \forall j = 1, \ldots, n$$
   (i.e. a sx troviam le risorse richieste da tutti gli oggetti assegnati al server $j$, a dx se $y_j = 1$ allora la capacità disponibile è $B$, se 0 allora il server non è attivo e quindi non può ospitare oggetti)
3. (chiaramente) $x_{ij}, y_j \in \{0, 1\}$

Questo modello matematico è un esempio di **Integer Linear Programming (ILP)** (ottimizzazione lineare con variabili intere), un problema NP-hard.

In questo senso, per risolverlo in tempo utile, si possono sfruttare diverse **euristiche di placement**:
- **Random**: sceglie un server u.a.r. e verifica che lo spazio sia sufficiente. Se sì inserisci l'oggetto, altriemtni ripeti.
- **Round-Robin**: scorri i server in ordine, mantenendo la posizione e inserendo l'oggetto nel primo che ha spazio sufficiente.
- **First Fit**: uguale a RR ma per ogni oggetto si riparte dal primo server
- **First Fit Decreasing**: ordina gli oggetti dal più grande al più piccolo e poi applica First Fit. Collocare prima gli oggetti grandi pemette di evitare la frammentazione dello spazio in tanti piccoli residui (che può comportare nei casi peggiori di non riuscire a collocare tutti gli oggetti, anche se una soluzione esiste), lo svantaggio è che ordinare costa.
- **Best Fit**: per ogni oggetto, scegli il server che lascerebbe meno spazio residuo (ma sufficiente, $\ge 0$) dopo l'inserimento.
- **Worst Fit**: per ogni oggetto, scegli il server che lascerebbe più spazio residuo dopo l'inserimento.

Le prime euristiche sono sicuramente meno sofisticate, ma molto veloci. Come al solito quindi la scelta dell'euristica dipende dal contesto e dai vincoli di tempo.

### MAPE: Execute
Qui si approfondisce di meno: la fase di execute infatti consiste solo nell'**attuare effettivamente le decisioni prese dal planner** (nel caso di placement ad esempio significa avviare/ricollocare le VM/container sui nodi scelti). 

Se il componente è già in esecuzione, allora il piano può richiedere una **(re)migrazione**: vedremo diverse delle tecniche di migrazione live nella parte di corso dedicata alla virtualizzazione. In generale comunque l'obiettivo è **mantenere la continuità del servizio riducendo al minimo l'interruzione**.

### Pattern MAPE
In questa sezione ci si chiede **dove eseguire le varie fasi del ciclo MAPE**, i.e. su quali nodi vengono eseguiti Monitor, Analyze, Plan e Execute; e come comunicano tra loro? (stiamo quindi affrontando l'architettura del managing system, non dell'applicazione gestita managed system).

Due modelli architetturali principali, al solito:
- *MAPE centralizzato*: ogni componente viene collocata su un *singolo* nodo dedicato. Architettura semplice, ma quel nodo può diventare **collo di bottiglia** oltre che **SPOF**.  
In particolare risulta difficile in *sistemi geograficamente distribuiti*: fare Monitoring con un'architettura del genere significherebbe che tutti i "sensori" debbano inviare i dati a un unico nodo centrale --> latenza significativa...
- **MAPE decentralizzato**: le componenti sono distrubuite su più nodi.  
Esistono a loro volta diversi pattern architetturali in questa soluzione, e come al solito nessuna è necessariamente migliore (dipende dal sistema, obiettivi, contesto..). Due pattern principali:
  - a **organizzazione gerarchica**: higher-level control loop manages subordinated control loops (**Master-Worker/Slave, Hierarchical Control**)
  - a **organizzazione flat**, con tutti i controllori che cooperano tra loro come peer (**Coordinated Control, Information Sharing**)

(NB i pattern per decentralizzare MAPE sono domanda frequente agli esoneri/esami)

<img src="imgs/patterns.png" width="400">

#### Patter gerarchico 
**Master-Slave**

In questo pattern la distribuzione delle responsabilità è **Master == Analyze + Plan** e **Slaves == Monitor + Execute**.

<img src="imgs/master-slave.png" width="400">

I **vari** worker (dislocati potenzialmente anche in diverse aree geografiche) raccolgono e possono pre-elaborare i dati locali, inviando al **singolo** master le informazioni già aggregate. Allora il master analizza la situazione, decide e invia ai worker execute le eventuali azioni da compiere.

Con questo approccio il **vantaggio** è che *il decisore ha una vista globale della situazione* e può quindi perseguire gli obiettivi dell'intero sistema.  
I **limiti** sono le *comunicazioni verso il master*, in quanto rappresentano un possibile bottleneck e dipendenza dal suo funzionamento oltre che SPOF.

Per evitare il problema di SPOF e migliorare quindi la disponibilità, ciò che si fa tipicamente è **replicare il master, con però un solo leader attivo alla volta**. In caso di guasto si sfrutta quindi un protocollo di **leader election** per decidere un nuovo master tra i nodi replicati.

**Hierarchical Control**

In quest'altro pattern gerarchico si hanno **più cicli MAPE completi, organizzati in una gerarchia.**  
Ogni livello ha responsabilità diverse, e può lavorare con frequenze diverse (es. il livello più basso può avere un ciclo MAPE molto veloce, mentre quello più alto può avere un ciclo più lento e quindi meno preciso ma con una visione più globale).

<img src="imgs/hierarchical.png" width="500">

es. *autoscaling di un'applicazione a microservizi* (la vedremo meglio poi).  
Qui ipotizziamo che controllori locali gestiscano lo scaling dei singoli microservizi, mentre un controllore superiore considera l'intera applicazione per coordinare le decisioni. 

Perché serve coordinare dall'alto? Perché un microservizio potrebbe considerare localmente valido aumentare le proprie capacità inviando molto più traffico ai servizi a valle, sovraccaricandoli --> una decisione locale può creare problemi altrove.  
Mentre i controllori locali possono intervenire frequentemente, quello globale può fornire direttive in tempi più lunghi.

Con questo pattern il **vantaggio** è la *separazione delle responsabilità in livelli diversi* e *maggiore flessibilità* (in quanto ogni livello può avere un ciclo MAPE con frequenza diversa).  
Il **limite** è *riuscire ad individuare una buona suddivisione dei livelli*: non tutti i sistemi si prestano facilemnte alla distinzione tra controllo locale e globale.

#### Pattern flat
**Coordinated Control**

In questo pattern ogni controllore segue un proprio ciclo MAPE su una parte del sistema. **Non c'è un superiore a livello gerarchico**: i cicli si coordinano attraverso interazioni tra le loro componenti --> ciò permette di distribuire il controllo senza concentrare le decisioni in un solo nodo.

<img src="imgs/flat.png" width="500">

Il **vantaggio** è *maggiore potenziale scalabilità* (infatti non c'è più un singolo master che può diventare collo di bottiglia all'aumentare del numero di controllori).  
Il **limite** è che *coordinare le decisioni tra tanti controllori allo stesso livello è complesso e ha un costo significativo*.

**Information Sharing**

Rappresenta un caso particolare di Coordinated Control, in cui **solo le componenti Monitor interagiscono tra loro**. Ha senso in quanto sono i Monitor che raccolgono i dati grezzi ed eventualmente li pre-elaborano.

I controllori condividono quindi le informazioni osservate via Monitoring a livello globale, ma ognuno di essi esegue separatamente Analyze, Plan ed Execute. 

<img src="imgs/info-sharing.png" width="500">

Il **vantaggio** è *minori interazioni* rispetto a un coordinamento esteso a tutte le fasi, il **limite** è che i planner (non avendo una visione globale dei dati nella loro interezza) *possono prendere decisioni sub-ottimali o in conflitto*.

Si vedono ora 3 esempi di sistemi auto-adattivi per la gestione risorse, al fine di avere chiari questi pattern di controllo (da studiare per l'esonero!).

In tutti e tre i casi le applicazioni operano "under uncertainty" (i.e. possibili cambiamenti nel carico di lavoro, guasti dei nodi etc.) e l'adaptation goal è mantenere i target di qualità SLO (es. tempo di risposta, availability, cost..)

#### Caso d'uso 1: EC2 Auto Scaling
Si vogliono adattare il numero di istanze EC2, a questo scopo si usa un servizio AWS chiamato **Amazon EC2 Auto Scaling**.     
Un **Auto Scaling Group** è un insieme di istanze gestite insieme dal servizio: si stabiliscono un minimo e un massimo e il servizio modifica le capacità entro tali limiti, secondo le politiche configurate.  
Il massimo in particolare serve a limitare la crescita dei costi.

Le azioni da considerare sono **scale-out** (aggiungere istanze) e **scale-in** (rimuovere istanze), si tratta di **scaling orizzontale**.

<img src="imgs/autoscalingEC2.png" width="400">

Per il **Monitoring** la componente si appoggia a **CloudWatch**: le metriche di risorse in questo caso sono uso CPU, memoria, disco, rete e conteggio delle richieste tramite un Application Load Balancer.  
In generale è da ricordare quindi che **la decisione di scaling deve usare una metrica che sia rappresentativa del carico o delle prestazioni**.

(ricorda che si distinguono in generale due tipi di metriche: **metriche di risorse** (CPU, memoria, disco, rete) e **metriche applicative specifiche** (es. lunghezza delle code, sessioni attive, tempi di risposta..). Per le metriche applicative serve *strumentazione*: bisogna predisporre codice o strumenti che le misurino/rendino disponibili per il monitoraggio.  
In generale non bisogna mai assumere che alcuna misura citata sia disponibile senza configurazione adeguata)

Il servizio Auto Scaling permette anche **self-healing**: sostituire istanze considerate non sane attraverso i controlli configurati --> es. se la capacità è di 5 istanze e una si guasta, allora il sistema ne trova una sostitutiva per mantenere la capacità prevista.  
L'operazione di self-healing è però distinta da quella di autoscaling: la prima riguarda il recupero dal guasto mentre la seconda l'adattamento al carico di lavoro.

Per il **Planning** studiamo due delle più semplici policy implementate in EC2 Auto Scaling per lo scale-in e lo scale-out.  

##### Policy reattiva a soglie

La prima è una semplice **politica reattiva a soglie**: si tratta di una semplice euristica basata su due soglie:
$$\begin{cases} \text{scale-out} & \text{se la metrica supera la soglia superiore} \\ \text{scale-in} & \text{se la metrica scende sotto la soglia inferiore} \\ \text{nessuna modifica} & \text{altrimenti}\end{cases}$$
Quindi l'utente deve definire precisamente le *soglie*, le *metriche*, le *finestre temporali di valutazione* e il *numero di istanze da aggiungere/rimuovere nei due casi*.

es. Utilizzo medio della CPU > 70% per 1 minuto --> scale-out di un'istanza
es. Utilizzo medio della CPU < 30% per 5 minuti --> scale-in di un'istanza

La politica è **reattiva** perché decide sulla base di **condizioni già osservate**. Tra le due soglie non ci si aspettano cambiamenti per evitare continue aggiunte e rimozioni di istanze.

Nel modello MAPE, sarebbe: Monitor raccogliere e aggregare la metrica in questione, Analyze rileva il superamento della soglia, Plan applica la regola per determinare la variazione di capacità ed Execute effettua l'azione di scaling.

Ad ogni azione di scaling effettuata, c'è un intervallo di tempo di **cooldown**, introdotto per lasciare al sistema il tempo di assestarsi e per ossservare l'effetto dell'intervento prima di avviarne eventualmente altri.  
Il tempo di cooldown è diverso dalla **finestra di valutazione**, che invece è il periodo di tempo in cui si osserva la metrica per stabilire se la condizione di scaling si sia o meno verificata.

Si distinguono in questa politica reattiva a soglia:
- **simple scaling** se al verificarsi della condizione si applica una variazione fissa configurata, seguita dal periodo di cooldown
- **step scaling** se invece la variazione dipende da quanto la metrica supera la soglia (es. se la CPU > 70% allora aggiungi 1 istanza, se > 80% allora aggiungi 2 istanze etc..).

Nonostante la policy reattiva a soglie sia molto semplice, ci sono diversi svantaggi:
1. **è difficile scegliere una metrica significativa e serve conoscere l'applicazione specifica**: un'applicazione può essere limitata da molteplici fattori come CPU, memoria, I/O... se ci si limita ad osservarne solo alcuni allora si rischia di non reagire correttamente
2. **è difficile scegliere soglie e tempi corretti** (vedi grafici tra poco): se ad esempio
   - *Soglia superiore troppo alta* --> rischio di aggiungere le risorse troppo tardi e quindi peggioramento prestazioni
   - *Soglia superiore troppo bassa* --> rischio di aggiungere risorse troppo presto e quindi spreco di risorse
   - *Soglia inferiore troppo alta* --> rischio di rimuovere le risorse in modo aggressivo e quindi possibile capacità insufficiente
   - *Soglia inferiore troppo bassa* --> rischio di rimuovere le risorse troppo tardi e quindi spreco di risorse
   - *Tempi di valutazione o cooldown troppo lunghi* --> reazione lenta ai cambiamenti
3. **è tendenzialmente inadatto per carichi di lavoro fortemente variabili**

Qui di seguito alcuni grafici che mostrano configurazioni in termini di threshold errate:

<img src="imgs/autoecw.png" width="600">


Nei grafici a sx le richieste sono in rosso e il tempo di risposta in nero.  
In quelli a dx numero di istanze sulle ordinate e il tempo ascisse.  

Il primo esperimento mostra **capacità eccessiva dopo un adattamento tardivo** (over-provisioning): la configurazione è che ogni 30 secondi controllo scaling, con soglia alta 80% e soglia bassa 25%.  
Si osserva dal grafico come il tempo di risposta alle richieste (nero) presenti vari picchi, sta a significare che *il sistema sta aggiungendo capacità troppo lentamente rispetto al bisogno* (ciò è contribuito all'uso di una high_thr eccessivamente alto prima di decidere di aggiungere risorse...).  
Successivamente il numero di istanze raggiunge 9 e resta alto, prima di diminuire gradualmente fino a 5. Questa fase, prima di stabilizzarsi a 5, è proprio l'**over-provisioning** essendo mantenuta più capacità del necessario (a causa di una low_thr evidentemente troppo bassa..) --> politica troppo lenta ad aggiungere e troppo lenta a rimuovere risorse!

Il secondo esperimento riguarda invece una **rimozione troppo aggressiva**: configurazione con periodo di 60 secondi, soglia alta 80% e soglia bassa 50%.  
Qui si osserva come il traffico di richieste (rosso) diminuisce, ma ad un tratto il tempo di risposta (nero) aumenta, questo è proprio dovuto al fatto che la *capacità viene ridotta più velocemente di quanto sia opportuno rispetto al carico residuo*.  
Il grafico delle istanze mostra una discesa fino ad una singola istanza che risulta insufficiente, producendo una fase di **under-previsioning**.

Per risolvere questi problemi un'idea è **sfruttare il reinforcement learning per imparare le soglie**.  
Ad es. si potrebbe adottare una soluzione multiagente con uno che si occupa della soglia relativa alla CPU, l'altro per la memoria, con l'obiettivo di adattare il comportamento a carichi che possono colpire risorse diverse. 

Il punto è che quindi **se si decide di utilizzare una policy di controllo a soglie --> si deve conoscere bene l'applicazione per capire come configurare le soglie e i tempi di valutazione**.

##### Policy reattiva Target Tracking
**Questa policy prevede, invece di specificare due soglie, di indicare un valore obiettivo**, come "mantenere l'utilizzo medio della CPU attorno al 60%" --> il controllore modifica il numero di istanze per avvicinarsi a quel valore.

Rimane una politica reattiva in quanto parte da una metrica osservata, e non da una previsione.  
I vantaggi di questa policy sono una **maggiore semplicità di configurazione** (non serve scegliere due soglie e tempi di valutazione, ma solo un target per ogni metrica) e la rilevanza nell'industria (Kubernetes HPA).

Kubernetes HPA (Horizontal Pod Autoscaler) è un esempio di policy target tracking che segue la regola:
$$N_{desired} = \lceil\left(N_{current} \times \frac{M_{current}}{M_{target}}\right)\rceil$$
dove $N$ è il numero di istanze e $M$ il valore della metrica osservata.  

Ad esempio con 5 repliche, CPU media all'80% e target al 60%:
$$N_{desired} = \lceil(5 \times \frac{80}{60})\rceil = \lceil(6.67)\rceil = 7$$
per cui il controllore propone di passare da 5 a 7 repliche.  

Intuitivamente ciò funziona perché il lavoro può essere distribuito tra più repliche se eccessivo --> la formula va bene fintanto che la metrica funziona in modo adeguato alla variazione di capacità, **non va applicata indistintamente a qualunque misura**.

##### Predictive scaling
Lo scaling predittivo usa un modello ML per prevedere il carico e preparare la capacità **prima che si presenti il bisogno** --> è un approccio **proattivo**. 

La motivazione è il ritardo nell'avvio delle risorse: se sappiamo che il traffico aumenterà --> possiamo avviare le istanze abbastanza presto per averle pronte.

Tra i limiti ci sono:
- servono dati storici rappresentativi
- una variazione improvvisa e mai osservata può essere difficile da prevedere
- supporta solo politiche di scale-out
- si assume che tutte le istanze VM siano equivalenti

Infatti il predictive scaling considerato **aggiunge capacità, ma non rimuove automaticamente quella diventata superflua: per lo scale-in serve una politica dinamica separata**. 

<img src="imgs/bbb.png" width="500">

**NB. L'autoscaling è argomento frequente nelle prove d'esame**

#### Caso d'uso 2: **selezionare dinamicamente i servizi**
Qui l'adattamento cambia forma: **non scegliamo più quante istanze avviare, ma quali servizi concreti utilizzare per realizzare un'applicazione**. 

Vediamo l'esempio di un'applicazione SOA (Service Oriented Architecture) composta da più servizi. Distinguiamo:
- **servizio astratto** == funzione richiesta dall'applicazione
- **servizio concreto** == specifica implementazione che fornisce quella funzione

<img src="imgs/soa.png" width="400">

In figura il servizio astratto $S_4$ ha più alternative concrete $s_{41}, s_{42}, s_{43}, s_{44}$ che offrono tutte funzionalità compatibili, ma possono essere diverse per **proprietà non funzionali** (es. tempo di risposta, disponibilità, costo, sicurezza..).

Il problema che ci si pone allora è *quali alternative scegliere al runtime per rispettare gli obiettivi di qualità dell'applicazione (SLAs)*?  

<img src="imgs/sa2.png" width="600">

A sinistra c'è la logica dell'applicazione, composta dai servizi $S_1, ..., S_6$. Si ricordano i tre costrutti **sequenza** (prima si sceglie un servizio e poi il successivo, in figura ad es. $S_1$ precede $S_2$), **scelta condizionale** (si sceglie il ramo in base a una condizione come if-else) e **fork-join** per cui più rami vengono eseguiti in parallelo, dove il ricongiungimento deve attendere il ramo più lento. Inoltre una parte del workflow può anche essere ripetuto (ciclo). 

Questa struttura conta in quanto la qualità globale dipende sia dai singoli servizi che da come sono composti, ad esempio in una sequenza i tempi si accumulano e se sono parte di un fork-join il tempo complessivo dipende dal ramo più lento.

**Il Planning per selezionare i servizi** viene in questo caso effettuato tramite *MOSES* (un planner centralizzato basato su programmazione lineare).  
Il planner considera caratteristiche e prestazioni dei servizi disponibili, la struttura dell'applicazione, i requisiti degli utenti ed eventuali classi di servizio diverse (es. una classe di utenti che paga di più può privilegiare in tempo di risposta). 

La scelta del planner di basarsi su formulazione lineare con variabili continue evita la difficoltà della formulazione intera vista nel bin packing, permettendo una risoluzione più gestibile al crescere del problema.

In generale non è necessario assegnare sempre un servizio astratto alla solita implementazione: ad esempio la politica può stabilire 50% delle invocazioni a una prima alternativa, 30% a una seconda e 20% a una terza --> il planner può decidere **come distribuire le richieste ai servizi concreti**.

Un'ulteriore possibilità è **invocare diversi servizi concreti in parallelo** e usare il primo che risponde. *Questo riduce i tempi di risposta medi, ma richiede più lavoro e risorse in quanto non sempre le invocazioni rimanenti si possono/riescono ad annullare.*  
Importante distinguere questo dal fork-join del workflow in quanto il fork-join prevede che i rami siano tutti necessari, mentre qui si parallelizza per questioni di prestazioni.

In questo caso quindi **MAPE** si vede come segue: 
- **Monitor e Analyze** come osservare prestazioni, disponibilità e rispetto obiettivi
- **Plan** come calcolare una nuova politica di selezione e coordinamento dei servizi concreti
- **Execute** come applicare la nuova politica calcolata 

Quindi qui in generale l'aspetto centrale è mantenere la funzione generale dell'applicazione (logica dei servizi astratti vista prima) adattando dinamicamente le implementazioni concrete dei servizi.

#### Caso d'uso 3: **autoscaling gerarchico dei microservizi**
Questo caso si applica all'**hierarchical control** spiegato precedentemente. 

L'obiettivo è **mantenere il tempo di risposta sotto un certo limite**, adattando il numero di repliche dei microservizi che compongono l'applicazione.

Il sistema ha due livelli: **Controllori locali (Microservice Manager)** e **Controllore globale (Application Manager)**.

Per quanto riguarda i Controllori locali, ogni microservizio ha un proprio controllore che:
- osserva il suo funzionamento
- valuta le necessità di adattamento
- produce una proposta di scale-in/scale-out per il microservizio che gestisce

In questo senso il Planning locale combina **teoria delle code** per modellare il servizio delle richieste e **previsione del carico**, realizzata in figura tramite regressione lineare. La proposta di adattamento viene quindi inviata al controllore superioere (globale).

<img src="imgs/hierarchical2.png" width="600">

Il Controllore globale ha una vista dell'intera applicazione e può approvare o rifiutare le proposte locali. Le decisioni accettate vengono quindi trasmesse alle componenti Execute locali.  

Il controllore globale serve perché i microservizi sono tra loro tutti collegati --> una richiesta di $A$ può generare lavoro per $B$ (es. $A$ invia richieste a $B$ e intanto il controllore di $A$ chiede di aumentare le repliche per inviare più richieste, mentre quello di $B$ propone di diminuirle --> conflitto: $B$ rischia di non avere capacità sufficiente per servire le richieste di $A$. Allora il controllore globale decide quale delle due proposte accettare).

Nota come questo sia hierarchical control e non master-worker perché i livelli locali hanno tutti un ciclo MAPE completo, e non solo Monitor+Execute.

<img src="imgs/ezempi.png" width="500">